这种方式需要按照
JSON Schema规范拼接JSON字符串，比较繁琐，并且缺少校验机制。不推荐。
举例1：返回简单结构
模型初始化：

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}  #关闭思考模式
)

In [5]:
json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        }, "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    }, "required": ["title", "year", "director", "rating"]}

structured_model = model.with_structured_output(
    json_schema,
    method="json_schema"
)
response = structured_model.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}
<class 'dict'>


说明：
1、method：结构化输出的方式，但是否可用，依赖于模型供应商及Langchain适配器的具体实现。比
如，DeepSeek模型服务不支持json_shema模式。
json_schema：使用模型供应商提供的专用结构化输出功能。
2、以上代码中定义json_schema的时候指定的title、description、type、properties、required是遵循
JSON Schema 规范的标准关键字，是固定写法。几个关键字的解释如下：
title ：为整个 Schema 或特定属性提供一个人类可读的标题，不能是中文，用于提高可读性。
description ：提供更详细的文字描述，说明 Schema 或属性的用途等，和 title一样，旨在帮助理
解。
type ：定义当前数据节点必须是什么数据类型。常见类型有 string, number, integer, boolean,
object, array, null。object即是json对象。
properties ：用于定义JSON 对象（Object）中可以包含哪些属性（键），以及每个属性对应的值
类型和说明。
required ：当 type为 "object"时使用，是一个数组，列出了对象中必须存在的属性名。

举例2：返回嵌套结构

In [7]:
"""
使用 JSON Schema 定义嵌套结构
"""
# 1. 定义嵌套的 JSON Schema
project_schema = {
    "title": "MovieInfo",
    "description": "包含电影标题、上映年份、导演、演员和评分的电影对象",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "电影标题"},

        "year": {"type": "integer", "description": "上映年份"},
        "director": {"type": "string", "description": "导演"},
        "cast": {  # 定义嵌套数组
            "type": "array",
            "description": "演员列表",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "演员姓名"},
                    "role": {"type": "string", "description": "演员角色"}
                },
                "required": ["name", "role"]
            }
        },
        "rating": {"type": "number", "description": "评分（10分制）"}
    },
    "required": ["title", "year", "director", "cast", "rating"]
}
# 绑定 JSON Schema 到模型
structured_model = model.with_structured_output(project_schema, method="json_schema")
# 调用模型
response = structured_model.invoke("生成一个关于《星际穿越》的电影信息，包含导演、演员、评分")
print(response)

{'title': '星际穿越', 'year': 2014, 'director': '克里斯托弗·诺兰', 'cast': [{'name': '马修·麦康纳', 'role': '库珀'}, {'name': '安妮·海瑟薇', 'role': '布兰德博士'}, {'name': '杰西卡·查斯坦', 'role': '墨菲·库珀'}, {'name': '迈克尔·凯恩', 'role': '布兰德教授'}, {'name': '卡西·阿弗莱克', 'role': '唐纳德'}, {'name': '麦肯吉·弗依', 'role': '少年墨菲'}, {'name': '蒂莫西·柴勒梅德', 'role': '汤姆·库珀'}], 'rating': 9.4}
